In [40]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas kagglehub

import os
import pandas as pd
import pgserver

from sqlalchemy import create_engine, text

pg = pgserver.get_server("/tmp/hw5_pg", cleanup_mode="stop")

engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())


Note: you may need to restart the kernel to use updated packages.
PostgreSQL 16.2 on aarch64-apple-darwin23.5.0, compiled by Apple clang version 15.0.0 (clang-1500.0.40.1), 64-bit


In [41]:
%load_ext sql

%config SqlMagic.autopandas = True

%sql $engine.url


The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [42]:
import kagglehub

dataset_path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

print("Dataset path:", dataset_path)
print("Files:", os.listdir(dataset_path))


Dataset path: /Users/macbook/.cache/kagglehub/datasets/olistbr/brazilian-ecommerce/versions/2
Files: ['olist_sellers_dataset.csv', 'product_category_name_translation.csv', 'olist_orders_dataset.csv', 'olist_order_items_dataset.csv', 'olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_products_dataset.csv']


In [43]:
import os
import kagglehub

# Завантажуємо датасет Olist
dataset_dir = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

print("Dataset directory:", dataset_dir)
print("\nAvailable CSV files:")

for file in sorted(os.listdir(dataset_dir)):
    if file.endswith(".csv"):
        print(file)


Dataset directory: /Users/macbook/.cache/kagglehub/datasets/olistbr/brazilian-ecommerce/versions/2

Available CSV files:
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [44]:
files = {
    "olist_customers_raw": "olist_customers_dataset.csv",
    "olist_orders_raw": "olist_orders_dataset.csv",
    "olist_order_items_raw": "olist_order_items_dataset.csv",
    "olist_products_raw": "olist_products_dataset.csv",
    "olist_sellers_raw": "olist_sellers_dataset.csv",
}

for table_name, file_name in files.items():
    path = os.path.join(dataset_dir, file_name)

    df = pd.read_csv(path)

    df.to_sql(table_name, engine, if_exists="replace", index=False, chunksize=10_000)

    print(table_name, df.shape)


olist_customers_raw (99441, 5)
olist_orders_raw (99441, 8)
olist_order_items_raw (112650, 7)
olist_products_raw (32951, 9)
olist_sellers_raw (3095, 4)


In [ ]:

%%sql

DROP TABLE IF EXISTS olist_order_items CASCADE;
DROP TABLE IF EXISTS olist_orders CASCADE;
DROP TABLE IF EXISTS olist_customers CASCADE;
DROP TABLE IF EXISTS olist_products CASCADE;
DROP TABLE IF EXISTS olist_sellers CASCADE;

CREATE TABLE olist_customers (
    customer_id TEXT PRIMARY KEY,
    customer_unique_id TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city TEXT,
    customer_state CHAR(2) NOT NULL
);

CREATE TABLE olist_orders (
    order_id TEXT PRIMARY KEY,
    customer_id TEXT NOT NULL
        REFERENCES olist_customers(customer_id),
    order_status TEXT NOT NULL,
    order_purchase_timestamp TIMESTAMP NOT NULL,
    order_approved_at TIMESTAMP,
    order_delivered_carrier_date TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_products (
    product_id TEXT PRIMARY KEY,
    product_category_name TEXT,
    product_name_length INTEGER
        CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER
        CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty INTEGER
        CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g INTEGER
        CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm INTEGER
        CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm INTEGER
        CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm INTEGER
        CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city TEXT,
    seller_state CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id TEXT NOT NULL
        REFERENCES olist_orders(order_id),
    order_item_id INTEGER NOT NULL,
    product_id TEXT
        REFERENCES olist_products(product_id),
    seller_id TEXT
        REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price NUMERIC(12, 2)
        CHECK (price IS NULL OR price >= 0),
    freight_value NUMERIC(12, 2)
        CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);


 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw5_pg
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.
Done.


""


In [46]:

%%sql

INSERT INTO olist_customers
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix,
    customer_city,
    customer_state
FROM olist_customers_raw;

INSERT INTO olist_products
SELECT
    product_id,
    product_category_name,
    product_name_lenght,
    product_description_lenght,
    product_photos_qty,
    product_weight_g,
    product_length_cm,
    product_height_cm,
    product_width_cm
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT
    seller_id,
    seller_zip_code_prefix,
    seller_city,
    seller_state
FROM olist_sellers_raw;

INSERT INTO olist_orders
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP,
    order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_order_items
SELECT
    order_id,
    order_item_id,
    product_id,
    seller_id,
    shipping_limit_date::TIMESTAMP,
    price,
    freight_value
FROM olist_order_items_raw;


 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw5_pg
99441 rows affected.
32951 rows affected.
3095 rows affected.
99441 rows affected.
112650 rows affected.


""


In [47]:

%%sql

CREATE TABLE IF NOT EXISTS hw5_referrals (
    referrer_customer_id TEXT NOT NULL,
    referred_customer_id TEXT NOT NULL,
    PRIMARY KEY (referrer_customer_id, referred_customer_id),
    FOREIGN KEY (referrer_customer_id)
        REFERENCES olist_customers(customer_id),
    FOREIGN KEY (referred_customer_id)
        REFERENCES olist_customers(customer_id),
    CHECK (referrer_customer_id <> referred_customer_id)
);


 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw5_pg
Done.


""


Завдання 2. CTE-chain pipeline з ≥5 stages

Побудуйте feature engineering pipeline через CTE-chain із щонайменше п’яти named stages. Кожен stage має мати короткий коментар у форматі -- Stage N: ... і виконувати одну логічну трансформацію.

In [55]:

%%sql

WITH
-- Stage 0: stable reference date for historical dataset
reference_date AS (
    SELECT
        MAX(order_purchase_timestamp)::DATE + 1 AS as_of_date
    FROM olist_orders
),

-- Stage 1: calculate total amount per order
stage1_orders_total AS (
    SELECT
        c.customer_unique_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(
            SUM(
                COALESCE(oi.price, 0)
                + COALESCE(oi.freight_value, 0)
            ), 0
        )::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        c.customer_unique_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),

-- Stage 2: calculate RFM features per unique customer
stage2_rfm AS (
    SELECT
        s.customer_unique_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER
            AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY
        s.customer_unique_id,
        rd.as_of_date
),

-- Stage 3: calculate customer cohort quarter
stage3_cohort AS (
    SELECT
        r.*,
        DATE_TRUNC(
            'quarter',
            r.first_order_date
        )::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),

-- Stage 4: calculate rolling 30-day average order value
stage4_rolling_aov AS (
    SELECT
        s.customer_unique_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_unique_id
            ORDER BY s.order_ts
            RANGE BETWEEN
                INTERVAL '30 days' PRECEDING
                AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_unique_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),

-- Stage 5: calculate cancellation rate per customer
stage5_cancel_rate AS (
    SELECT
        customer_unique_id,
        AVG(
            CASE
                WHEN order_status = 'canceled'
                THEN 1.0
                ELSE 0.0
            END
        )::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_unique_id
),

-- Stage 6: calculate category revenue per customer
stage6_category_revenue AS (
    SELECT
        c.customer_unique_id,
        COALESCE(
            p.product_category_name,
            '(unknown)'
        ) AS product_category_name,
        SUM(
            COALESCE(oi.price, 0)
            + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY
        c.customer_unique_id,
        COALESCE(
            p.product_category_name,
            '(unknown)'
        )
),

-- Stage 7: select top category by revenue
stage7_preferred_category AS (
    SELECT
        customer_unique_id,
        product_category_name,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_unique_id
                ORDER BY
                    cr.category_revenue DESC,
                    cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)

-- Final output: one row per unique customer
SELECT
    c.customer_unique_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(
        aov.avg_order_value_30d, 0
    )::NUMERIC(12, 2) AS avg_order_value_30d_recent,
    COALESCE(
        cancel.cancellation_rate, 0
    )::NUMERIC(6, 4) AS cancellation_rate,
    COALESCE(
        pc.product_category_name,
        '(none)'
    ) AS preferred_category,
    COALESCE(
        pc.category_revenue, 0
    )::NUMERIC(12, 2) AS preferred_category_revenue

FROM (
    SELECT DISTINCT ON (customer_unique_id)
        customer_unique_id,
        customer_state
    FROM olist_customers
    ORDER BY customer_unique_id, customer_id DESC
) AS c

LEFT JOIN stage3_cohort AS coh
    ON coh.customer_unique_id = c.customer_unique_id

LEFT JOIN stage4_rolling_aov AS aov
    ON aov.customer_unique_id = c.customer_unique_id
   AND aov.rn_recent = 1

LEFT JOIN stage5_cancel_rate AS cancel
    ON cancel.customer_unique_id = c.customer_unique_id

LEFT JOIN stage7_preferred_category AS pc
    ON pc.customer_unique_id = c.customer_unique_id

ORDER BY monetary DESC NULLS LAST, c.customer_unique_id
LIMIT 20;




 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw5_pg
20 rows affected.


,customer_unique_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
0,0a0a92112bd4c708ca5fde585afaa872,RJ,384,1,13664.08,2017-09-29,2017-07-01,13664.08,0.0000,telefonia_fixa,13664.08
1,da122df9eeddfedc1dc1f5349a1a690c,RJ,565,2,7571.63,2017-04-01,2017-04-01,3785.82,0.0000,eletroportateis,7571.63
2,763c8b1c9c68a0229c42c9fc6f662b93,ES,95,1,7274.88,2018-07-15,2018-07-01,7274.88,0.0000,telefonia_fixa,7274.88
3,dc4802a71eae9be1dd28f5d788ceb526,MS,613,1,6929.31,2017-02-12,2017-01-01,6929.31,0.0000,utilidades_domesticas,6929.31
4,459bef486812aa25204be022145caa62,ES,85,1,6922.21,2018-07-25,2018-07-01,6922.21,0.0000,pcs,6922.21
5,ff4159b92c40ebe40454e3e6a7c35ed6,SP,512,1,6726.66,2017-05-24,2017-04-01,6726.66,0.0000,artes,6726.66
6,4007669dec559734d6f53e029e360987,MG,328,1,6081.54,2017-11-24,2017-10-01,6081.54,0.0000,agro_industria_e_comercio,6081.54
7,5d0a2980b292d049061542014e8960bf,GO,98,1,4809.44,2018-07-12,2018-07-01,4809.44,1.0000,cool_stuff,4809.44
8,eebb5dda148d3893cdaf5b5ca3040ccb,SP,548,1,4764.34,2017-04-18,2017-04-01,4764.34,0.0000,eletroportateis,4764.34
9,48e1ac109decbb87765a3eade6854098,PB,118,1,4681.78,2018-06-22,2018-04-01,4681.78,0.0000,pcs,4681.78


### Завдання 2. CTE-chain Feature Engineering Pipeline

**Business-question:** Як сформувати набір поведінкових ознак покупців Olist для подальшої сегментації та використання в ML-моделях?

**Реалізація:**

Побудовано CTE-chain pipeline із восьми named stages (`Stage 0–7`). Кожен етап виконує окрему логічну трансформацію даних.

Pipeline розраховує RFM-ознаки (`recency_days`, `frequency`, `monetary`), квартал першої покупки (`cohort_quarter`), ковзну середню вартість замовлень за 30 днів, частку скасованих замовлень і найпопулярнішу категорію товарів за revenue.

Для розрахунку `recency_days` використано стабільну `reference_date`, визначену на основі максимальної дати замовлення в датасеті.

Для customer-level features використано `customer_unique_id`, щоб об'єднати всі замовлення одного покупця.

Window functions `AVG() OVER (...)` та `ROW_NUMBER()` застосовано для розрахунку ковзного середнього і вибору найпопулярнішої категорії.

**Business insight:**

Отримані ознаки можна використовувати для RFM-сегментації, прогнозування відтоку клієнтів (churn prediction), персоналізованих рекомендацій і побудови моделей прогнозування цінності покупців (Customer Lifetime Value).

Розділення логіки на CTE-етапи спрощує перевірку правильності обчислень, підтримку та повторне використання pipeline.

Завдання 3. Recursive CTE для referral chain

Olist не містить referral-таблиці. Створіть синтетичну hw5_referrals — пари referrer_customer_id → referred_customer_id — і використайте recursive CTE, щоб обчислити depth і path у referral-tree.

In [57]:

%%sql

DROP TABLE IF EXISTS hw5_referrals CASCADE;

CREATE TABLE hw5_referrals (
    referrer_customer_id TEXT NOT NULL
        REFERENCES olist_customers(customer_id),

    referred_customer_id TEXT NOT NULL
        REFERENCES olist_customers(customer_id),

    referred_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),

    PRIMARY KEY (referred_customer_id),

    CHECK (referrer_customer_id <> referred_customer_id)
);

WITH ordered_customers AS (
    SELECT
        customer_id,
        ROW_NUMBER() OVER (
            ORDER BY customer_id
        ) AS rn
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 10
)

INSERT INTO hw5_referrals (
    referrer_customer_id,
    referred_customer_id
)
SELECT
    prev.customer_id,
    nxt.customer_id
FROM ordered_customers AS prev
JOIN ordered_customers AS nxt
    ON nxt.rn = prev.rn + 1
ON CONFLICT (referred_customer_id) DO NOTHING;


 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw5_pg
Done.
Done.
9 rows affected.


""


In [58]:

%%sql

SELECT
    referrer_customer_id,
    referred_customer_id,
    referred_at
FROM hw5_referrals
ORDER BY referrer_customer_id;


 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw5_pg
9 rows affected.


,referrer_customer_id,referred_customer_id,referred_at
0,00012a2ce6f8dcda20d059ce98491703,000161a058600d5901f007fab4c27140,2026-11-05 23:29:58.105725+01:00
1,000161a058600d5901f007fab4c27140,0001fd6190edaaf884bcaf3d49edf079,2026-11-05 23:29:58.105725+01:00
2,0001fd6190edaaf884bcaf3d49edf079,0002414f95344307404f0ace7a26f1d5,2026-11-05 23:29:58.105725+01:00
3,0002414f95344307404f0ace7a26f1d5,000379cdec625522490c315e70c7a9fb,2026-11-05 23:29:58.105725+01:00
4,000379cdec625522490c315e70c7a9fb,0004164d20a9e969af783496f3408652,2026-11-05 23:29:58.105725+01:00
5,0004164d20a9e969af783496f3408652,000419c5494106c306a97b5635748086,2026-11-05 23:29:58.105725+01:00
6,000419c5494106c306a97b5635748086,00046a560d407e99b969756e0b10f282,2026-11-05 23:29:58.105725+01:00
7,00046a560d407e99b969756e0b10f282,00050bf6e01e69d5c0fd612f1bcfb69c,2026-11-05 23:29:58.105725+01:00
8,00050bf6e01e69d5c0fd612f1bcfb69c,000598caf2ef4117407665ac33275130,2026-11-05 23:29:58.105725+01:00


In [59]:

%%sql

WITH RECURSIVE referral_depth AS (

    -- ANCHOR: root customers who were not referred
    SELECT
        c.customer_id,
        1 AS depth,
        ARRAY[c.customer_id]::TEXT[] AS path_ids,
        c.customer_id::TEXT AS path_text
    FROM olist_customers AS c
    WHERE NOT EXISTS (
        SELECT 1
        FROM hw5_referrals AS r
        WHERE r.referred_customer_id = c.customer_id
    )

    UNION ALL

    -- RECURSIVE: move from referrer to referred customer
    SELECT
        r.referred_customer_id AS customer_id,
        rd.depth + 1 AS depth,
        rd.path_ids || r.referred_customer_id AS path_ids,
        rd.path_text || ' -> ' || r.referred_customer_id AS path_text
    FROM hw5_referrals AS r
    JOIN referral_depth AS rd
        ON rd.customer_id = r.referrer_customer_id
    WHERE NOT (r.referred_customer_id = ANY(rd.path_ids))
      AND rd.depth < 50
)

SELECT
    depth,
    COUNT(*) AS n_customers,
    MAX(path_text) AS sample_path
FROM referral_depth
GROUP BY depth
ORDER BY depth;


 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw5_pg
10 rows affected.


,depth,n_customers,sample_path
0,1,99432,ffffe8b65bbe3087b653a978c870db99
1,2,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
2,3,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
3,4,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
4,5,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
5,6,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
6,7,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
7,8,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
8,9,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
9,10,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...


### Reflection — Recursive CTE та захист від циклів

Для аналізу синтетичного referral chain використано `WITH RECURSIVE`, який складається з початкової частини (`ANCHOR`) та рекурсивної частини (`RECURSIVE`).

`ANCHOR` визначає покупців, які не були запрошені іншими учасниками, а рекурсивна частина послідовно знаходить наступних покупців у ланцюжку. На кожному кроці збільшується `depth` і формується повний шлях `path`.

Якщо у referral-графі виникне цикл `A → B → A`, без захисту рекурсивний запит може нескінченно повторювати переходи між покупцями, витрачаючи ресурси бази даних.

Для запобігання цьому використано два механізми:

1. **`path_ids` та `ANY()`** — масив зберігає вже відвідані `customer_id`, а перевірка `NOT (next_id = ANY(path_ids))` не дозволяє повторно відвідати того самого покупця.
2. **Depth guard** — умова `rd.depth < 50` обмежує максимальну глибину рекурсії до 50 рівнів.

У PostgreSQL також можна використовувати `CYCLE` clause для автоматичного виявлення циклів.

Ці механізми важливі для production-систем, оскільки захищають рекурсивні запити від некоректних циклічних зв'язків і неконтрольованого виконання.

Завдання 4. Materialized View з CONCURRENT REFRESH

Оберніть повний результат CTE-pipeline з Завдання 2 у Materialized View. MV має містити ті самі ключові features, а не скорочену версію pipeline. Після створення MV додайте UNIQUE INDEX на customer_id і виконайте REFRESH MATERIALIZED VIEW CONCURRENTLY.

In [63]:

%%sql

DROP MATERIALIZED VIEW IF EXISTS mv_olist_customer_features CASCADE;

CREATE MATERIALIZED VIEW mv_olist_customer_features AS

WITH
-- Stage 0: stable reference date for historical dataset
reference_date AS (
    SELECT
        MAX(order_purchase_timestamp)::DATE + 1 AS as_of_date
    FROM olist_orders
),

-- Stage 1: calculate total amount per order
stage1_orders_total AS (
    SELECT
        c.customer_unique_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(
            SUM(
                COALESCE(oi.price, 0)
                + COALESCE(oi.freight_value, 0)
            ), 0
        )::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        c.customer_unique_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),

-- Stage 2: RFM features per unique customer
stage2_rfm AS (
    SELECT
        s.customer_unique_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER
            AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY
        s.customer_unique_id,
        rd.as_of_date
),

-- Stage 3: cohort quarter from first order date
stage3_cohort AS (
    SELECT
        r.*,
        DATE_TRUNC(
            'quarter',
            r.first_order_date
        )::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),

-- Stage 4: rolling 30-day average order value
stage4_rolling_aov AS (
    SELECT
        s.customer_unique_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_unique_id
            ORDER BY s.order_ts
            RANGE BETWEEN
                INTERVAL '30 days' PRECEDING
                AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_unique_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),

-- Stage 5: cancellation rate per customer
stage5_cancel_rate AS (
    SELECT
        customer_unique_id,
        AVG(
            CASE
                WHEN order_status = 'canceled'
                THEN 1.0
                ELSE 0.0
            END
        )::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_unique_id
),

-- Stage 6: revenue by customer and category
stage6_category_revenue AS (
    SELECT
        c.customer_unique_id,
        COALESCE(
            p.product_category_name,
            '(unknown)'
        ) AS product_category_name,
        SUM(
            COALESCE(oi.price, 0)
            + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY
        c.customer_unique_id,
        COALESCE(
            p.product_category_name,
            '(unknown)'
        )
),

-- Stage 7: select preferred category
stage7_preferred_category AS (
    SELECT
        customer_unique_id,
        product_category_name,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_unique_id
                ORDER BY
                    cr.category_revenue DESC,
                    cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)

-- Final output: one row per unique customer
SELECT
    c.customer_unique_id,
    c.customer_state,

    COALESCE(coh.recency_days, 9999)
        AS recency_days,

    COALESCE(coh.frequency, 0)
        AS frequency,

    COALESCE(coh.monetary, 0)::NUMERIC(12, 2)
        AS monetary,

    coh.first_order_date,
    coh.cohort_quarter,

    COALESCE(
        aov.avg_order_value_30d, 0
    )::NUMERIC(12, 2)
        AS avg_order_value_30d_recent,

    COALESCE(
        cancel.cancellation_rate, 0
    )::NUMERIC(6, 4)
        AS cancellation_rate,

    COALESCE(
        pc.product_category_name, '(none)'
    ) AS preferred_category,

    COALESCE(
        pc.category_revenue, 0
    )::NUMERIC(12, 2)
        AS preferred_category_revenue

FROM (
    SELECT DISTINCT ON (customer_unique_id)
        customer_unique_id,
        customer_state
    FROM olist_customers
    ORDER BY customer_unique_id, customer_id DESC
) AS c

LEFT JOIN stage3_cohort AS coh
    ON coh.customer_unique_id = c.customer_unique_id

LEFT JOIN stage4_rolling_aov AS aov
    ON aov.customer_unique_id = c.customer_unique_id
   AND aov.rn_recent = 1

LEFT JOIN stage5_cancel_rate AS cancel
    ON cancel.customer_unique_id = c.customer_unique_id

LEFT JOIN stage7_preferred_category AS pc
    ON pc.customer_unique_id = c.customer_unique_id;


 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw5_pg
Done.
96096 rows affected.


""


In [64]:
%%sql

CREATE UNIQUE INDEX idx_mv_olist_customer_features_unique_id
ON mv_olist_customer_features (customer_unique_id);



 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw5_pg
Done.


""


In [65]:

%%sql

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT customer_unique_id) AS unique_customers,
    COUNT(*) - COUNT(DISTINCT customer_unique_id) AS duplicates
FROM mv_olist_customer_features;


 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw5_pg
1 rows affected.


,total_rows,unique_customers,duplicates
0,96096,96096,0


In [66]:

%%sql

SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST, customer_unique_id
LIMIT 5;


 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw5_pg
5 rows affected.


,customer_unique_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
0,0a0a92112bd4c708ca5fde585afaa872,RJ,384,1,13664.08,2017-09-29,2017-07-01,13664.08,0.0000,telefonia_fixa,13664.08
1,da122df9eeddfedc1dc1f5349a1a690c,RJ,565,2,7571.63,2017-04-01,2017-04-01,3785.82,0.0000,eletroportateis,7571.63
2,763c8b1c9c68a0229c42c9fc6f662b93,ES,95,1,7274.88,2018-07-15,2018-07-01,7274.88,0.0000,telefonia_fixa,7274.88
3,dc4802a71eae9be1dd28f5d788ceb526,MS,613,1,6929.31,2017-02-12,2017-01-01,6929.31,0.0000,utilidades_domesticas,6929.31
4,459bef486812aa25204be022145caa62,ES,85,1,6922.21,2018-07-25,2018-07-01,6922.21,0.0000,pcs,6922.21


In [67]:

%%sql

REFRESH MATERIALIZED VIEW CONCURRENTLY
mv_olist_customer_features;


 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw5_pg
Done.


""


In [68]:

%%sql

SELECT
    indexname,
    indexdef
FROM pg_indexes
WHERE tablename = 'mv_olist_customer_features';


 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw5_pg
1 rows affected.


,indexname,indexdef
0,idx_mv_olist_customer_features_unique_id,CREATE UNIQUE INDEX idx_mv_olist_customer_feat...


### Завдання 4. Materialized View з Concurrent Refresh

Для збереження результатів feature engineering pipeline створено Materialized View `mv_olist_customer_features` на основі повного CTE-chain із завдання 2.

Materialized View містить RFM-ознаки (`recency_days`, `frequency`, `monetary`), квартал першої покупки (`cohort_quarter`), середню вартість замовлень за 30 днів, частку скасованих замовлень (`cancellation_rate`) та найпопулярнішу категорію товарів (`preferred_category`).

Як ключ використано `customer_unique_id`, оскільки він ідентифікує унікального покупця та дозволяє об'єднати історію його замовлень. Це забезпечує правильну гранулярність customer-level ML-features.

Для підтримки `REFRESH MATERIALIZED VIEW CONCURRENTLY` створено унікальний індекс на `customer_unique_id`.

Materialized View дозволяє зберігати попередньо обчислені ознаки та використовувати їх в аналітиці й ML-моделях без повторного виконання складного SQL-pipeline.

`REFRESH MATERIALIZED VIEW CONCURRENTLY` забезпечує оновлення даних зі збереженням доступності попередньої версії MV для читання.

Завдання 5. EXPLAIN ANALYZE + Reflection

Запустіть EXPLAIN (ANALYZE, BUFFERS) на двох сценаріях: повне переобчислення CTE-pipeline і lookup із Materialized View. Збережіть обидва output у notebook.



In [69]:

%%sql


EXPLAIN (ANALYZE, BUFFERS)

WITH
-- Stage 0: stable reference date for historical dataset
reference_date AS (
    SELECT
        MAX(order_purchase_timestamp)::DATE + 1 AS as_of_date
    FROM olist_orders
),

-- Stage 1: calculate total amount per order
stage1_orders_total AS (
    SELECT
        c.customer_unique_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(
            SUM(
                COALESCE(oi.price, 0)
                + COALESCE(oi.freight_value, 0)
            ), 0
        )::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        c.customer_unique_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),

-- Stage 2: RFM features per unique customer
stage2_rfm AS (
    SELECT
        s.customer_unique_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER
            AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY
        s.customer_unique_id,
        rd.as_of_date
),

-- Stage 3: cohort quarter from first order date
stage3_cohort AS (
    SELECT
        r.*,
        DATE_TRUNC(
            'quarter',
            r.first_order_date
        )::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),

-- Stage 4: rolling 30-day average order value
stage4_rolling_aov AS (
    SELECT
        s.customer_unique_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_unique_id
            ORDER BY s.order_ts
            RANGE BETWEEN
                INTERVAL '30 days' PRECEDING
                AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_unique_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),

-- Stage 5: cancellation rate per customer
stage5_cancel_rate AS (
    SELECT
        customer_unique_id,
        AVG(
            CASE
                WHEN order_status = 'canceled'
                THEN 1.0
                ELSE 0.0
            END
        )::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_unique_id
),

-- Stage 6: revenue by customer and category
stage6_category_revenue AS (
    SELECT
        c.customer_unique_id,
        COALESCE(
            p.product_category_name,
            '(unknown)'
        ) AS product_category_name,
        SUM(
            COALESCE(oi.price, 0)
            + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY
        c.customer_unique_id,
        COALESCE(
            p.product_category_name,
            '(unknown)'
        )
),

-- Stage 7: select preferred category
stage7_preferred_category AS (
    SELECT
        customer_unique_id,
        product_category_name,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_unique_id
                ORDER BY
                    cr.category_revenue DESC,
                    cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)

-- Final output: one row per unique customer
SELECT
    c.customer_unique_id,
    c.customer_state,

    COALESCE(coh.recency_days, 9999)
        AS recency_days,

    COALESCE(coh.frequency, 0)
        AS frequency,

    COALESCE(coh.monetary, 0)::NUMERIC(12, 2)
        AS monetary,

    coh.first_order_date,
    coh.cohort_quarter,

    COALESCE(
        aov.avg_order_value_30d, 0
    )::NUMERIC(12, 2)
        AS avg_order_value_30d_recent,

    COALESCE(
        cancel.cancellation_rate, 0
    )::NUMERIC(6, 4)
        AS cancellation_rate,

    COALESCE(
        pc.product_category_name, '(none)'
    ) AS preferred_category,

    COALESCE(
        pc.category_revenue, 0
    )::NUMERIC(12, 2)
        AS preferred_category_revenue

FROM (
    SELECT DISTINCT ON (customer_unique_id)
        customer_unique_id,
        customer_state
    FROM olist_customers
    ORDER BY customer_unique_id, customer_id DESC
) AS c

LEFT JOIN stage3_cohort AS coh
    ON coh.customer_unique_id = c.customer_unique_id

LEFT JOIN stage4_rolling_aov AS aov
    ON aov.customer_unique_id = c.customer_unique_id
   AND aov.rn_recent = 1

LEFT JOIN stage5_cancel_rate AS cancel
    ON cancel.customer_unique_id = c.customer_unique_id

LEFT JOIN stage7_preferred_category AS pc
    ON pc.customer_unique_id = c.customer_unique_id;


 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw5_pg
165 rows affected.


,QUERY PLAN
0,Merge Left Join (cost=93701.34..131341.41 row...
1,Merge Cond: (olist_customers.customer_unique...
2,"Buffers: shared hit=15348, temp read=11385 w..."
3,CTE stage1_orders_total
4,-> Finalize GroupAggregate (cost=18904.8...
...,...
160,...
161,Planning:
162,Buffers: shared hit=59
163,Planning Time: 7.046 ms


2. EXPLAIN для lookup із Materialized View
Тут перевіримо, наскільки швидко PostgreSQL знаходить ознаки одного покупця через UNIQUE INDEX (customer_unique_id).

In [70]:

%%sql

EXPLAIN (ANALYZE, BUFFERS)

WITH target_customer AS (
    SELECT customer_unique_id
    FROM mv_olist_customer_features
    ORDER BY customer_unique_id
    LIMIT 1
)

SELECT mv.*
FROM mv_olist_customer_features AS mv
JOIN target_customer AS t
    ON t.customer_unique_id = mv.customer_unique_id;


 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw5_pg
14 rows affected.


,QUERY PLAN
0,Nested Loop (cost=0.83..8.90 rows=1 width=92)...
1,Buffers: shared hit=8
2,-> Limit (cost=0.42..0.46 rows=1 width=33)...
3,Buffers: shared hit=4
4,-> Index Only Scan using idx_mv_olist...
5,Heap Fetches: 0
6,Buffers: shared hit=4
7,-> Index Scan using idx_mv_olist_customer_f...
8,Index Cond: (customer_unique_id = mv_o...
9,Buffers: shared hit=4


In [71]:

%%sql

SELECT
    indexname,
    indexdef
FROM pg_indexes
WHERE tablename = 'mv_olist_customer_features';


 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw5_pg
1 rows affected.


,indexname,indexdef
0,idx_mv_olist_customer_features_unique_id,CREATE UNIQUE INDEX idx_mv_olist_customer_feat...


Top-10 покупців за Monetary

In [72]:

%%sql

CREATE INDEX IF NOT EXISTS idx_mv_olist_customer_features_monetary
ON mv_olist_customer_features (monetary DESC NULLS LAST);


 * postgresql://postgres:***@/postgres?host=%2Fprivate%2Ftmp%2Fhw5_pg
Done.


""


## Reflection — CTE Pipeline, Materialized View та Recursive CTE

**1. Найдорожчий feature у pipeline**

Найбільш ресурсомісткими є обчислення `avg_order_value_30d_recent` та `preferred_category`, оскільки вони потребують обробки великої кількості замовлень, агрегації через `GROUP BY` і сортування для window functions (`AVG OVER`, `ROW_NUMBER`). У моєму `EXPLAIN ANALYZE` видно `Merge Left Join`, `Finalize GroupAggregate` та значне використання буферів і тимчасових файлів. Повне переобчислення pipeline займає **1353.054 ms**, а `Planning Time` становить 7.046 ms. Без детального аналізу всіх вузлів плану неможливо однозначно визначити найдорожчий feature.

**2. Чи варто використовувати Materialized View?**

Так, оскільки обчислення всіх ознак займає понад секунду. Для customer-level ML-features я б обрала **щоденне оновлення** через `REFRESH MATERIALIZED VIEW CONCURRENTLY`. Це дозволяє зберігати попередньо обчислені результати та забезпечувати доступність даних під час оновлення.

**3. Коли звичайний VIEW кращий?**

Звичайний `VIEW` доцільніший, якщо потрібні максимально актуальні дані, наприклад для моніторингу нових замовлень або змін їхнього статусу в реальному часі. Він не потребує окремого refresh, оскільки виконує запит до базових таблиць під час звернення.

**4. Як Recursive CTE забезпечує termination?**

Рекурсія завершується, коли більше немає нових рядків. Для захисту від циклів використано масив `path_ids` із перевіркою `NOT (next_id = ANY(path_ids))` та обмеження `depth < 50`. Якщо виникне цикл `A → B → A`, повторний перехід до `A` буде заблоковано.

**5. Найважливіший trade-off**

Найважливішим вважаю баланс між **latency та freshness**. Lookup із MV займає лише **0.158 ms**, використовує `Index Only Scan` і `Index Scan` та потребує `shared hit=8`. Це значно менше за 1353.054 ms повного pipeline, хоча сценарії виконують різний обсяг роботи. Щоденний refresh забезпечує прийнятну актуальність даних і швидкий доступ до ознак без постійного переобчислення.